# Stage 1 — 10x glomerular segmentation

Resolve one session, build per-odor correlation maps, curate a shared mask, and extract traces.


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, "..")

from analysis.session.devshim import LocalGroup
from analysis.session.resolve import resolve_group
from analysis.session.zscore import make_group_keys
from analysis.session.corrcache import build_group_correlation_maps
from analysis.seg_10x.watershed import GLOM_10X_DEFAULTS, scale_params

GROUP_ID = 198
MANIPULATION = ""
APPROVED_ONLY = False
EXCLUDE_ACQ = []
SEPARATE_BY_CONDITION = False
DETREND = True
PRE_S, POST_S, SIGMA_PX = 2.0, 3.0, 0.5

# Keep large temporary writes local; set these roots once per machine.
SCRATCH = Path(os.environ.get("ODYN_SCRATCH_ROOT", Path.home() / "odyn_scratch"))
MAIN = Path(os.environ.get("ODYN_IMAGING_ROOT", "/Volumes/MossLab/ImagingData"))
SCRATCH.mkdir(parents=True, exist_ok=True)

# A local metadata copy avoids repeated slow reads from the server. The resolver
# uses frame sync when available and otherwise the best acquisition timing source.
group = LocalGroup(
    MAIN / ".odyn" / "odyn.db", MAIN,
    snapshot_to=SCRATCH / "odyn_snapshot.db", max_age_s=1800,
)
session = resolve_group(
    group, group_id=GROUP_ID, manipulation=MANIPULATION,
    approved_only=APPROVED_ONLY, exclude_acq_ids=tuple(EXCLUDE_ACQ),
)
print(session.summary())


## Correlation maps

Each trial is baseline-z-scored, averaged within odor group, and reduced to an 8-neighbour local-correlation map. Temporary accumulators stay on local scratch because repeated server writes are slow; only the compact maps are cached with the session.


In [ ]:
keys = make_group_keys(
    session.odor_ids, session.states,
    separate_by_condition=SEPARATE_BY_CONDITION,
)
WORK = SCRATCH / f"nb_work_{GROUP_ID}"
CORR_CACHE = session.output_dir / "correlation_cache"

corr_by_odor, corr_meta = build_group_correlation_maps(
    session.paths,
    odor_on_frames=session.odor_on_frames,
    odor_off_frames=session.odor_off_frames,
    group_keys=keys,
    frame_rate=session.frame_rate,
    pre_s=PRE_S,
    post_s=POST_S,
    spatial_sigma_px=SIGMA_PX,
    work_dir=WORK,
    cache_dir=CORR_CACHE,
)
print(f"{len(corr_by_odor)} maps; cache {corr_meta['cache']}")


## Segmentation GUI

The GUI moves through three states:

1. **Tune** — adjust segmentation for all odors or override one odor. Threshold controls sensitivity; adaptive thresholding handles uneven backgrounds; diameter limits set ROI size; peak distance controls watershed splitting; border excludes edge artifacts.
2. **Merge** — freeze segmentation and combine matching detections. Minimum overlap controls matching, minimum detections requires support across odors, and consensus fraction controls how much of the overlapping footprint is retained.
3. **Curate** — freeze parameters, then add, delete, or exclude ROIs on the merged mask. Returning to an earlier state discards later edits.

Manual additions grow from watershed seeds. **Save masks + config** stores the curated mask and settings.


In [ ]:
from analysis.seg_10x.gui import launch

params = scale_params(GLOM_10X_DEFAULTS, to_um_per_px=session.um_per_px)
gui = launch(
    corr_by_odor,
    save_path=SCRATCH / f"masks_group{GROUP_ID}.npz",
    params=params,
)


## Final mask

Run after GUI curation. A saved mask is reused after a kernel restart; otherwise the current GUI state supplies the mask.


In [ ]:
from analysis.session.finalize import mask_hash
from analysis.session.masks import (
    background_image, load_latest_mask, save_mask_overlay, save_masks_mat,
)
from analysis.session.store import session_filename

saved = load_latest_mask(session.output_dir)
if "gui" in dir() and gui.state.phase == "curate":
    labels = gui.state.curated_mask()
    source = "curated"
elif saved is not None:
    labels = saved["labels"]
    source = f"saved: {saved['path'].name}"
else:
    labels = gui.state.merged().labels
    source = "automatic"

masks_by_group = gui.state.segment_all() if "gui" in dir() else {}
order = list(masks_by_group)
masks = list(masks_by_group.values())
params = dict(gui.state.shared) if "gui" in dir() else params

reference = background_image(corr_by_odor)
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(reference, cmap="gray", vmin=np.percentile(reference, 1),
          vmax=np.percentile(reference, 99.5))
ax.contour(labels > 0, levels=[0.5], colors="magenta", linewidths=0.6)
ax.set(title=f"{int(labels.max())} ROIs — {source}", xticks=[], yticks=[])

def output_path(kind, suffix):
    return session.output_dir / session_filename(
        group_id=session.group_id, exp_name=session.exp_name,
        kind=kind, suffix=suffix,
    )

mat = save_masks_mat(
    output_path("masks", ".mat"), labels,
    per_group_masks=masks_by_group,
    exp_name=session.exp_name,
    group_id=session.group_id,
    mask_hash=mask_hash(labels),
)
png = save_mask_overlay(output_path("masks", ".png"), reference, labels)
print(mat.name, png.name)


## Trace extraction

Apply the final mask to every acquisition and write the session HDF5 plus compact QC outputs. The build uses local scratch and copies the completed file to the server, reducing slow writes and allowing interrupted trial extraction to resume.


In [ ]:
RUN_EXTRACTION = True

from analysis.session.finalize import finalize_session, mask_hash, verify
from analysis.session.store import read_session

existing = verify(session.output_dir)
current_hash = mask_hash(labels)
already_current = existing["status"] == "ok" and existing["mask_hash"] == current_hash

round_path = None
if RUN_EXTRACTION and not already_current:
    result = finalize_session(
        session, labels,
        per_group_masks={repr(k): m for k, m in masks_by_group.items()},
        segmentation_params=params,
        merge_params=dict(gui.state.merge_params) if "gui" in dir() else {},
        curation=gui.state.summary() if "gui" in dir() and gui.state.phase == "curate" else None,
        images=corr_by_odor,
        pre_s=PRE_S,
        post_s=POST_S,
        neuropil=True,
        scratch_dir=SCRATCH,
        detrend=DETREND,
    )
    round_path = Path(result["path"])
elif RUN_EXTRACTION and already_current:
    round_path = session.output_dir / existing["file"]

if round_path is not None:
    from analysis.session.response_qc import response_qc

    qc = response_qc(round_path)
    print(round_path.name, qc["figure"])
    for flag in qc["flags"]:
        print(f"! {flag}")
